# Chapter 04 - pandas로 데이터에 질문하기

## 질문
**어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?**

- CSV: customers, products, orders, order_items
- 금액: `quantity × unit_price`
- 범위: `order_status == "completed"`
- 집계: 카테고리 / 상품 / 월 / 고객

> Evidence 1: `images/step01_question.png`


## STEP 1~2. 환경, 데이터, 실제 컬럼 확인


In [2]:
import sys
from pathlib import Path
import pandas as pd

current = Path.cwd().resolve()
project_root = current
while project_root != project_root.parent and not (project_root / "data").is_dir():
    project_root = project_root.parent
if not (project_root / "data").is_dir():
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")

DATA_DIR = project_root / "data" / "raw"
REPORT_DIR = project_root / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

customers = pd.read_csv(DATA_DIR / "customers.csv")
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")

datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
}

print("Python:", sys.executable)
print("프로젝트 루트:", project_root)
for name, df in datasets.items():
    print("\n", name, df.shape)
    print(df.columns.tolist())

print("\n[city]")
print(customers["city"].value_counts(dropna=False))
print("\n[order_status]")
print(orders["order_status"].value_counts(dropna=False))


Python: c:\Users\dkrak\AppData\Local\Programs\Python\Python314\python.exe
프로젝트 루트: C:\dev\llm-data-analysis-course

 customers (150, 6)
['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

 products (100, 4)
['product_id', 'product_name', 'category', 'price']

 orders (300, 5)
['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

 order_items (764, 5)
['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']

[city]
city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

[order_status]
order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64


### 확인
실제 컬럼명과 필터 값을 먼저 확인했다. 이후 코드는 이 출력 결과를 기준으로 작성한다.


## STEP 3~4. 필터링·정렬·`line_total`


In [3]:
customer_basic = customers[["customer_id", "gender", "age", "city"]].copy()
customers_over_30 = customers[customers["age"] >= 30].copy()
city_customers = customers[customers["city"].isin(["서울", "부산"])].copy()
top10_products = products.sort_values("price", ascending=False).head(10)

order_items = order_items.copy()
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

print("30세 이상:", len(customers_over_30))
print("서울/부산:", len(city_customers))
display(top10_products)
display(order_items[["order_id","product_id","quantity","unit_price","line_total"]].head(10))
print("전체 주문 상세 금액:", order_items["line_total"].sum())


30세 이상: 111
서울/부산: 31


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000
5,3,33,5,124000,620000
6,3,20,3,80000,240000
7,3,11,5,115000,575000
8,4,45,3,62000,186000
9,4,84,4,78000,312000


전체 주문 상세 금액: 255610000


### 해석
현재 `line_total` 합계에는 주문 상태가 반영되지 않았다. 따라서 아직 completed 주문 금액이라고 부르면 안 된다.

> Evidence 2: `images/step02_transform.png`


## STEP 5. orders merge 검증


In [4]:
print("orders.order_id 중복:", orders["order_id"].duplicated().sum())

order_sales = order_items.merge(
    orders[["order_id","customer_id","order_date","order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전:", len(order_items))
print("병합 후:", len(order_sales))
print(order_sales["_merge"].value_counts(dropna=False))

assert len(order_items) == len(order_sales)
assert (order_sales["_merge"] == "both").all()

order_sales = order_sales.drop(columns="_merge")


orders.order_id 중복: 0
병합 전: 764
병합 후: 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64


### 해석
오른쪽 `orders.order_id`가 고유한지 확인하고 `many_to_one`으로 병합했다. 병합 전후 행 수와 미매칭도 확인했다.

> Evidence 3: `images/step03_merge.png`


## STEP 6~7. completed 필터 + products merge


In [5]:
order_sales["order_date"] = pd.to_datetime(order_sales["order_date"], errors="coerce")
print("날짜 변환 실패:", order_sales["order_date"].isna().sum())

completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

completed_order_sales["order_month"] = (
    completed_order_sales["order_date"].dt.to_period("M").astype(str)
)

print(completed_order_sales["order_status"].value_counts(dropna=False))
print("products.product_id 중복:", products["product_id"].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("products 병합 전:", len(completed_order_sales))
print("products 병합 후:", len(completed_sales_items))
print(completed_sales_items["_merge"].value_counts(dropna=False))

assert len(completed_order_sales) == len(completed_sales_items)
assert (completed_sales_items["_merge"] == "both").all()

completed_sales_items = completed_sales_items.drop(columns="_merge")


날짜 변환 실패: 0
order_status
completed    474
Name: count, dtype: int64
products.product_id 중복: 0
products 병합 전: 474
products 병합 후: 474
_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64


### 해석
이후 `total_sales`는 completed 주문 상세의 `quantity × unit_price` 합계다. 회계상 순매출이나 실제 회사 매출이라고 단정하지 않는다.


## STEP 8~10. 카테고리·상품·월·고객 집계


In [6]:
category_sales = (
    completed_sales_items
    .groupby("category", as_index=False)
    .agg(total_quantity=("quantity","sum"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)
category_sales["sales_ratio"] = (
    category_sales["total_sales"] / category_sales["total_sales"].sum() * 100
).round(2)

product_sales = (
    completed_sales_items
    .groupby(["product_id","product_name","category"], as_index=False)
    .agg(total_quantity=("quantity","sum"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)

monthly_summary = (
    completed_order_sales
    .groupby("order_month", as_index=False)
    .agg(total_sales=("line_total","sum"), order_count=("order_id","nunique"))
    .sort_values("order_month")
)
monthly_summary["average_order_value"] = (
    monthly_summary["total_sales"] / monthly_summary["order_count"]
).round(0)

customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(order_count=("order_id","nunique"), total_sales=("line_total","sum"))
    .sort_values("total_sales", ascending=False)
)
customer_sales = customer_sales.merge(
    customers[["customer_id","city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)
customer_sales["customer_label"] = "Customer " + customer_sales["customer_id"].astype(str)

display(category_sales)
display(product_sales.head(10))
display(monthly_summary)
display(customer_sales[["customer_label","city","order_count","total_sales"]].head(10))

print("날짜 범위:", completed_order_sales["order_date"].min(),
      "~", completed_order_sales["order_date"].max())


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


날짜 범위: 2025-07-09 00:00:00 ~ 2026-07-05 00:00:00


### 해석
`total_sales`가 크다고 가장 인기 있거나 가장 수익성이 높다고 바로 단정할 수 없다. 판매 수량, 단가, 기간, 원가 등 추가 확인이 필요하다.

> Evidence 4: `images/step04_groupby.png`


## STEP 11. 총합 교차 검증


In [7]:
base_total = completed_order_sales["line_total"].sum()
checks = pd.Series({
    "base": base_total,
    "category": category_sales["total_sales"].sum(),
    "product": product_sales["total_sales"].sum(),
    "month": monthly_summary["total_sales"].sum(),
    "customer": customer_sales["total_sales"].sum(),
}, name="total_sales")

display(checks.to_frame())
print("총합 모두 일치:", checks.nunique() == 1)
assert checks.nunique() == 1


,total_sales
base,148990000
category,148990000
product,148990000
month,148990000
customer,148990000


총합 모두 일치: True


### 해석
같은 completed 범위의 여러 집계 총합이 원본과 일치하는지 확인했다. 이는 현재 확인 가능한 merge/groupby 중복·누락 가능성을 줄이는 검증이다.

> Evidence 5: `images/step05_total_check.png`


## STEP 12. 결과 CSV 저장 및 재확인


In [8]:
files = {
    "category": REPORT_DIR / "ch04_category_sales.csv",
    "product": REPORT_DIR / "ch04_product_sales.csv",
    "monthly": REPORT_DIR / "ch04_monthly_sales.csv",
    "customer": REPORT_DIR / "ch04_customer_sales.csv",
}

category_sales.to_csv(files["category"], index=False, encoding="utf-8-sig")
product_sales.to_csv(files["product"], index=False, encoding="utf-8-sig")
monthly_summary.to_csv(files["monthly"], index=False, encoding="utf-8-sig")
customer_sales.to_csv(files["customer"], index=False, encoding="utf-8-sig")

for name, path in files.items():
    print(name, path.exists(), path.stat().st_size)

saved = pd.read_csv(files["category"])
print(saved.shape)
print(saved.columns.tolist())
display(saved.head())


category True 254
product True 4409
monthly True 442
customer True 3387
(7, 4)
['category', 'total_quantity', 'total_sales', 'sales_ratio']


,category,total_quantity,total_sales,sales_ratio
0,스포츠,295,31743000,21.31
1,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
3,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12


## STEP 13. LLM 코드 검증


In [9]:
# ChatGPT 제안 코드: completed 주문 기준 월별 금액
llm_items = order_items[["order_id","quantity","unit_price"]].copy()
llm_items["line_total"] = llm_items["quantity"] * llm_items["unit_price"]

llm_sales = llm_items.merge(
    orders[["order_id","order_date","order_status"]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("병합 전/후:", len(llm_items), len(llm_sales))
print(llm_sales["_merge"].value_counts(dropna=False))

llm_sales["order_date"] = pd.to_datetime(llm_sales["order_date"], errors="coerce")
print("날짜 변환 실패:", llm_sales["order_date"].isna().sum())

llm_completed = llm_sales[llm_sales["order_status"] == "completed"].copy()
llm_completed["order_month"] = llm_completed["order_date"].dt.to_period("M").astype(str)

llm_monthly = (
    llm_completed
    .groupby("order_month", as_index=False)
    .agg(total_sales=("line_total","sum"), order_count=("order_id","nunique"))
    .sort_values("order_month")
)

compare = monthly_summary[["order_month","total_sales","order_count"]].merge(
    llm_monthly,
    on="order_month",
    how="outer",
    suffixes=("_mine","_llm"),
    indicator=True,
)
compare["sales_diff"] = compare["total_sales_mine"] - compare["total_sales_llm"]
compare["order_count_diff"] = compare["order_count_mine"] - compare["order_count_llm"]

display(compare)
print(
    "내 결과와 LLM 결과 일치:",
    (compare["_merge"] == "both").all()
    and (compare["sales_diff"] == 0).all()
    and (compare["order_count_diff"] == 0).all()
)


병합 전/후: 764 764
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
날짜 변환 실패: 0


,order_month,total_sales_mine,order_count_mine,total_sales_llm,order_count_llm,_merge,sales_diff,order_count_diff
0,2025-07,5869000,8,5869000,8,both,0,0
1,2025-08,15621000,18,15621000,18,both,0,0
2,2025-09,10190000,13,10190000,13,both,0,0
3,2025-10,25766000,26,25766000,26,both,0,0
4,2025-11,8812000,12,8812000,12,both,0,0
5,2025-12,11501000,14,11501000,14,both,0,0
6,2026-01,17423000,22,17423000,22,both,0,0
7,2026-02,9749000,17,9749000,17,both,0,0
8,2026-03,13429000,14,13429000,14,both,0,0
9,2026-04,17553000,23,17553000,23,both,0,0


내 결과와 LLM 결과 일치: True


### LLM 검토 결과
LLM 코드가 실행됐다는 사실만으로 정답이라고 보지 않았다. 실제 컬럼명, merge key, `many_to_one`, completed 범위, `order_id.nunique()`, 총합을 직접 비교했다.

최종 판단: **검증 후 사용**

> Evidence 6: `images/step06_llm_validation.png`


## 최종 인사이트용 수치


In [10]:
top_category = category_sales.iloc[0]
top_product = product_sales.iloc[0]
top_month = monthly_summary.loc[monthly_summary["total_sales"].idxmax()]

print("상위 카테고리:", top_category.to_dict())
print("상위 상품:", top_product.to_dict())
print("금액이 가장 큰 월:", top_month.to_dict())


상위 카테고리: {'category': '스포츠', 'total_quantity': 295, 'total_sales': 31743000, 'sales_ratio': 21.31}
상위 상품: {'product_id': 41, 'product_name': '스포츠 상품 041', 'category': '스포츠', 'total_quantity': 35, 'total_sales': 5705000}
금액이 가장 큰 월: {'order_month': '2025-10', 'total_sales': 25766000, 'order_count': 26, 'average_order_value': 991000.0}


## 최종 인사이트

1. 카테고리별 completed 주문 기준 금액에는 차이가 있다. 위 `category_sales`와 최종 수치 셀에서 가장 큰 카테고리를 확인한다.
2. 월별 금액은 주문 건수와 평균 주문 금액을 함께 봐야 한다. 단순히 월별 `total_sales`만 보고 증가·감소를 단정하지 않는다.

### 추가로 확인하고 싶은 질문
상위 카테고리의 금액이 큰 이유가 판매 수량 때문인지, 상품 단가 때문인지 더 확인하고 싶다.

### 현재 결과의 한계
현재 `total_sales`는 completed 주문의 `quantity × unit_price` 합계다. 할인, 배송비, 세금, 부분 환불, 원가와 마진을 반영한 회계상 순매출이나 이익은 아니다.
